In [1]:
import sys
import os

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import sqlite3

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from framework import init_env
init_env()

path = os.path.join("..", os.environ["DB_FILE"])
database = sqlite3.connect(path)

### Selecteer hier over welke editie dit rapport gaat

In [ ]:
edition = 2025

In [2]:
# Ophalen van de data
sql = """
SELECT O1.position, A.name, S.rowid AS song_id, S.title, S.year, (
	SELECT position
	FROM Occurance O2
	WHERE edition = {last}
	AND O1.song = O2.song
) AS pos_last, (
    SELECT COUNT(rowid)
    FROM OCCURANCE O3
    WHERE O3.edition <= {current} AND O1.song = O3.song
) As occurance_count
FROM OCCURANCE O1 
INNER JOIN Song S ON S.rowid = O1.song
INNER JOIN Artist A ON A.rowid = S.artist
WHERE O1.edition = {current}
ORDER BY O1.position ASC
""".format(last=edition - 1, current=edition)

df = pd.read_sql_query(sql, database).set_index("position")
df.describe()

,song_id,year,pos_last,occurance_count
count,2000.000000,2000.000000,1874.000000,2000.00000
mean,2600.510500,1992.303000,949.305229,16.75900
std,1508.559999,16.930372,556.908552,9.03763
min,1.000000,1956.000000,1.000000,1.00000
25%,1262.000000,1979.000000,469.250000,9.00000
50%,2675.000000,1990.000000,939.500000,17.00000
75%,3926.500000,2006.000000,1420.750000,27.00000
max,4933.000000,2025.000000,2000.000000,27.00000


# De top 10

In [3]:
df.head(10)

,name,song_id,title,year,pos_last,occurance_count
position,,,,,,
1,Queen,3135,Bohemian Rhapsody,1975,1.0,27
2,Billy Joel,420,Piano Man,1973,5.0,27
3,Eagles,1211,Hotel California,1977,3.0,27
4,Coldplay,835,Fix You,2005,2.0,17
5,Danny Vera,4708,Roller Coaster,2019,4.0,7
6,Pearl Jam,2996,Black,1992,10.0,17
7,Led Zeppelin,2295,Stairway To Heaven,1971,7.0,27
8,Boudewijn de Groot,573,Avond,1997,8.0,27
9,Golden Earring,1656,Radar Love (Albumversie),1973,20.0,27


# Grootse stijgers/zakkers

In [4]:
df['climb_rate'] = df['pos_last'] - df.index

## Grootste stijgers

In [5]:
df.sort_values(by=['climb_rate'], ascending=False).head(20)

,name,song_id,title,year,pos_last,occurance_count,climb_rate
position,,,,,,,
116,Marco Borsato,2496,Zij,2002,1566.0,14,1450.0
35,Marco Borsato,2501,De Waarheid,1996,1428.0,27,1393.0
82,Marco Borsato,2495,Afscheid Nemen Bestaat Niet,2003,1276.0,20,1194.0
93,Marco Borsato,2502,Ik Leef Niet Meer Voor Jou,1996,1238.0,26,1145.0
90,Marco Borsato,2498,De Bestemming,1998,1202.0,25,1112.0
830,The Beach Boys,3802,Sloop John B,1966,1931.0,27,1101.0
355,Benson Boone,4886,Beautiful Things,2024,1434.0,2,1079.0
182,"Marco Borsato, Armin van Buuren & Davina Michelle",4770,Hoe Het Danst,2019,1261.0,6,1079.0
910,Suzan & Freek,4737,Dromen In Kleur,2021,1962.0,4,1052.0


## Grootste zakkers

In [6]:
df.sort_values(by=['climb_rate'], ascending=True).head(20)

,name,song_id,title,year,pos_last,occurance_count,climb_rate
position,,,,,,,
1814,Queen,3171,It's A Beautiful Day,1995,644.0,5,-1170.0
1612,Claude,4856,Ladada (Mon Dernier Mot),2022,528.0,3,-1084.0
1958,Queen,3170,Good Old-Fashioned Lover Boy,1976,1035.0,6,-923.0
1879,Niall Horan,4664,Heaven,2023,1004.0,2,-875.0
1757,The Rolling Stones,4049,Brown Sugar,1971,884.0,27,-873.0
1993,Coldplay,846,Speed Of Sound,2005,1126.0,20,-867.0
1965,Taylor Swift,4553,You Need To Calm Down,2019,1101.0,3,-864.0
1877,David Bowie,1001,Changes,1971,1036.0,11,-841.0
1867,DI-RECT,1124,OMG It's Happening,2023,1071.0,2,-796.0


# Nieuwe binnenkomers

In [7]:
new_entries = df[df.pos_last.isnull()].drop(columns=['pos_last'])
print(new_entries['name'].count())

126


## Re-entries
De volgende nieuwe binnenkomers hebben al eens eerder in de top2000 gestaan: 

In [8]:
re_entries = new_entries[new_entries['occurance_count'] > 1]
row_count = len(re_entries.index)
print("Aantal re-entries: {}".format(row_count))
pd.set_option('display.max_rows', row_count)
display(re_entries)

Aantal re-entries: 53


,name,song_id,title,year,occurance_count,climb_rate
position,,,,,,
404,The Beach Boys,3803,Tears In The Morning,1970,24,NaN
872,Gerard Cox,1614,'T Is Weer Voorbij Die Mooie Zomer,1973,13,NaN
1288,Golden Earring,1676,The Devil Made Me Do It,1982,4,NaN
1442,Rob de Nijs,3321,Jan Klaassen De Trompetter,1973,15,NaN
1453,Marco Borsato,2506,Vrij Zijn,1995,8,NaN
1513,Suzan & Freek,4739,Honderd Keer,2022,2,NaN
1526,Suzan & Freek,4738,Kwijt,2022,2,NaN
1535,Suzan & Freek,4735,Weg Van Jou,2020,4,NaN
1558,Lionel Richie,2349,All Night Long (All Night),1983,19,NaN


## Echt nieuwe liedjes
De volgende nieuwe binnenkomers hebben nog nooit in de top2000 gestaan

In [9]:
real_new = new_entries[new_entries['occurance_count'] == 1].drop(columns=['occurance_count'])
row_count = len(real_new.index)
print("Aantal echt nieuwe liedjes: {}".format(row_count))
pd.set_option('display.max_rows', row_count)
display(real_new)

Aantal echt nieuwe liedjes: 73


,name,song_id,title,year,climb_rate
position,,,,,
33,Suzan & Freek,4741,Lichtje Branden,2021,NaN
153,René Karst,4913,Atje Voor De Sfeer,2019,NaN
306,Stef Ekkel & René Karst,4914,Liever Te Dik In De Kist Dan Weer Een Feestje ...,2019,NaN
311,Lola Young,4915,Messy,2024,NaN
357,Ozzy Osbourne,4630,Crazy Train,1980,NaN
391,Bente,4903,Kan Je Me Zien,2025,NaN
400,Suzan & Freek,4740,Niemand,2025,NaN
460,Radiohead,3222,Let Down,1997,NaN
473,Iron Maiden,1870,The Number Of The Beast,1982,NaN


# Verdwenen uit de lijst
De volgende liedjes komen dit jaar niet meer voor in de top2000. De lijst is gesorteerd op jaar van uitgaven van het liedje.

In [10]:
sql = """
SELECT O.position, A.name, S.title, S.year
FROM OCCURANCE O 
INNER JOIN Song S ON S.rowid = O.song
INNER JOIN Artist A ON A.rowid = S.artist
WHERE O.edition = {last}
AND O.song NOT IN (SELECT song FROM Occurance WHERE edition={current})
ORDER  BY S.year DESC
""".format(last=edition - 1, current=edition)

ex_entries = pd.read_sql_query(sql, database).set_index("position")
row_count = len(ex_entries.index)
print("Aantal uit de lijst verdwenen: {}".format(row_count))
pd.set_option('display.max_rows', row_count)
display(ex_entries)

Aantal uit de lijst verdwenen: 126


,name,title,year
position,,,
833,Macklemore,Hind's Hall,2024
1611,The Rolling Stones,Angry,2023
1161,Douwe Bob,This World Is Our Home,2023
1806,Post Malone,Chemical,2023
1870,Sera,Head Held High,2023
1817,P!nk,Never Gonna Not Dance Again,2022
1526,Harry Styles,Music For A Sushi Restaurant,2022
1899,Flemming,Paracetamollen,2022
1344,Rondé,Love Myself,2022
